# 🎬 MiniMax H3 视音频生成系统 · Google Colab 专属部署与云盘直存工作流

> **模型特点**：MiniMax H3（海螺 AI）多模态视音频生成模型，支持原生 2K/768p 24FPS 画面与 32kHz 立体声原生伴音。  
> **配置环境**：Google Colab（GPU: A100，建议开启 High-RAM）  
> **特色功能**：
> - 🚀 **ComfyUI 交互式前端**：支持文生视频（T2V）、图生视频（I2V）、参考生视频（R2V）。
> - 💾 **Google Drive 自动直存**：所有生成的 `.mp4` 视频及音频毫秒级同步到您的个人云端硬盘，断连不丢件。
> - ⚡ **4-Step Turbo-LoRA 极速推理**：A100 单卡 30~50 秒快速出片。
> - 📦 **云盘缓存机制（可选）**：支持将 38GB 模型归档在 Google Drive，下次启动无需重新下载。


### 步骤 1：GPU 与系统环境自检
检查是否已分配到 NVIDIA A100 GPU 以及高内存（High-RAM）环境。


In [1]:
# @title 1. GPU 与系统资源检查
import torch
import psutil
import os

print("="*60)
print("🔍 正在检查 Google Colab 硬件配置...")
print("="*60)

# 检查 GPU
if not torch.cuda.is_available():
    raise RuntimeError("❌ 未检测到 GPU！请在顶部菜单选择：代码执行程序 -> 更改运行时类型 -> 硬件加速器选择 GPU (A100)。")

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU 型号: {gpu_name}")
print(f"✅ GPU 显存: {total_vram:.2f} GB")

# 检查系统内存 (RAM)
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"✅ 系统内存 (RAM): {ram_gb:.2f} GB")

if ram_gb < 25:
    print("⚠️ 警告：当前未开启高内存（High-RAM）。建议在运行时设置中勾选 High-RAM 以获得最稳定的体验。")
else:
    print("🎉 恭喜！当前配置完全满足 MiniMax H3 的高速推理要求！")
print("="*60)


🔍 正在检查 Google Colab 硬件配置...
✅ GPU 型号: NVIDIA A100-SXM4-40GB
✅ GPU 显存: 39.49 GB
✅ 系统内存 (RAM): 83.47 GB
🎉 恭喜！当前配置完全满足 MiniMax H3 的高速推理要求！


### 步骤 2：挂载 Google Drive 并初始化专属存储目录
挂载云端硬盘后，系统将自动创建 `MiniMax_H3_Outputs` 文件夹，并配置可选的模型持久化缓存目录。


In [3]:
# @title 2. 挂载 Google 云端硬盘
from google.colab import drive
import os

print("📂 正在请求授权挂载 Google Drive...")
drive.mount('/content/drive')

# 定义 Google Drive 路径
GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_H3_Outputs"
GDRIVE_MODEL_CACHE = "/content/drive/MyDrive/MiniMax_Models"

os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(GDRIVE_MODEL_CACHE, exist_ok=True)

print("="*60)
print(f"✅ 视频自动保存路径已就绪：{GDRIVE_OUTPUT_DIR}")
print(f"✅ 模型持久化缓存路径已就绪：{GDRIVE_MODEL_CACHE}")
print("="*60)


📂 正在请求授权挂载 Google Drive...
Mounted at /content/drive
✅ 视频自动保存路径已就绪：/content/drive/MyDrive/MiniMax_H3_Outputs
✅ 模型持久化缓存路径已就绪：/content/drive/MyDrive/MiniMax_Models


### 步骤 3：安装 ComfyUI 核心环境与专属导演节点
部署 ComfyUI、ComfyUI-Manager 以及支持 MiniMax H3 多分段镜头控制的 `ComfyUI_MiniMaxH3_Director` 扩展。


In [4]:
# @title 3. 部署 ComfyUI 及扩展节点
import os
import subprocess

os.chdir('/content')

if not os.path.exists("/content/ComfyUI"):
    print("🚀 正在克隆 ComfyUI 官方代码库...")
    os.system("git clone https://github.com/comfyanonymous/ComfyUI.git")
else:
    print("ℹ️ ComfyUI 代码已存在，跳过克隆。")

os.chdir('/content/ComfyUI')
print("📦 正在安装 PyTorch 与核心加速依赖...")
os.system("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
os.system("pip install -q -r requirements.txt")
os.system("pip install -q accelerate diffusers huggingface_hub aria2 sentencepiece")

# 安装插件
os.chdir('/content/ComfyUI/custom_nodes')
if not os.path.exists("ComfyUI-Manager"):
    os.system("git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git")
if not os.path.exists("ComfyUI_MiniMaxH3_Director"):
    os.system("git clone -q https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git")

print("✅ ComfyUI 核心与扩展节点安装完毕！")


🚀 正在克隆 ComfyUI 官方代码库...
📦 正在安装 PyTorch 与核心加速依赖...
✅ ComfyUI 核心与扩展节点安装完毕！


### 步骤 4：高速下载 MiniMax H3 官方量化模型（支持 Drive 缓存复用）
下载专为 A100 优化的 FP8 骨干模型、NVFP4 文本/视觉编码器、4-Step Turbo-LoRA 与立体声音频 VAE。  
如果 Google Drive 中已有缓存，将自动建立软链接，无需重复下载 38GB 文件！


In [5]:
# @title 4. 模型智能下载与挂载（支持云盘秒级复用）
import os
import shutil
from huggingface_hub import hf_hub_download

REPO_ID = "Comfy-Org/MiniMax-H3"
LOCAL_MODELS_DIR = "/content/ComfyUI/models"
GDRIVE_MODEL_CACHE = "/content/drive/MyDrive/MiniMax_Models"

# 是否同步缓存到 Google Drive (True: 备份到云盘下次秒启; False: 仅存临时磁盘)
SAVE_MODELS_TO_GDRIVE = True # @param {type:"boolean"}

target_files = [
    # (子目录, 远程文件名)
    ("diffusion_models", "minimax_h3_fl2va_pruned_fp8_scaled.safetensors"),
    ("text_encoders", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("loras", "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"),
    ("vae", "minimax_h3_audio_vae_fp32.safetensors"),
    ("vae", "minimax_h3_video_vae_int8_convrot.safetensors")
]

print("="*60)
print("📥 正在处理 MiniMax H3 核心模型权重...")
print("="*60)

for sub_dir, fname in target_files:
    local_sub_path = os.path.join(LOCAL_MODELS_DIR, sub_dir)
    os.makedirs(local_sub_path, exist_ok=True)
    local_file = os.path.join(local_sub_path, fname)

    gdrive_sub_path = os.path.join(GDRIVE_MODEL_CACHE, sub_dir)
    os.makedirs(gdrive_sub_path, exist_ok=True)
    gdrive_file = os.path.join(gdrive_sub_path, fname)

    # 策略 1: Google Drive 中已有该模型，直接创建软链接（秒级完成）
    if os.path.exists(gdrive_file):
        print(f"⚡ [云盘复用] 检测到 Google Drive 缓存: {fname}")
        if not os.path.exists(local_file):
            os.symlink(gdrive_file, local_file)
        continue

    # 策略 2: 下载模型
    if not os.path.exists(local_file):
        print(f"🌐 [高速下载] 正在从 Hugging Face 下载: {fname} ...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=f"{sub_dir}/{fname}",
            local_dir=LOCAL_MODELS_DIR
        )

    # 策略 3: 若勾选备份到云盘，直接复制到 Google Drive 供下次使用
    if SAVE_MODELS_TO_GDRIVE and not os.path.exists(gdrive_file):
        print(f"💾 [云盘备份] 正在将 {fname} 同步备份到 Google Drive ...")
        shutil.copy2(local_file, gdrive_file)

print("="*60)
print("🎉 所有模型配置就绪！")
print("="*60)


📥 正在处理 MiniMax H3 核心模型权重...
⚡ [云盘复用] 检测到 Google Drive 缓存: minimax_h3_fl2va_pruned_fp8_scaled.safetensors
⚡ [云盘复用] 检测到 Google Drive 缓存: qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
⚡ [云盘复用] 检测到 Google Drive 缓存: minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors
⚡ [云盘复用] 检测到 Google Drive 缓存: minimax_h3_audio_vae_fp32.safetensors
⚡ [云盘复用] 检测到 Google Drive 缓存: minimax_h3_video_vae_int8_convrot.safetensors
🎉 所有模型配置就绪！


### 步骤 5：下载并注入预置工作流模版
将官方经过 A100 + Turbo-LoRA 调优的文生视频（T2V）与图生视频（I2V）工作流注入 ComfyUI。


In [6]:
# @title 5. 注入优化后的官方工作流模版
import urllib.request
import os

workflows_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(workflows_dir, exist_ok=True)

templates = {
    "minimax_h3_t2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json",
    "minimax_h3_i2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_i2v.json",
    "minimax_h3_r2v.json": "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_r2v.json"
}

for name, url in templates.items():
    dest = os.path.join(workflows_dir, name)
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"✅ 工作流模版已注入: {name}")
    except Exception as e:
        print(f"⚠️ 下载 {name} 失败: {e}")

print("🎨 工作流模版全部注入成功，可在 WebUI 的 Load 菜单中直接选择！")


✅ 工作流模版已注入: minimax_h3_t2v.json
✅ 工作流模版已注入: minimax_h3_i2v.json
✅ 工作流模版已注入: minimax_h3_r2v.json
🎨 工作流模版全部注入成功，可在 WebUI 的 Load 菜单中直接选择！


### 步骤 6：启动 ComfyUI 并连接 Cloudflare 安全公网隧道
服务启动后，ComfyUI 的输出目录将直接绑定到 Google Drive：
`/content/drive/MyDrive/MiniMax_H3_Outputs`  
在下方日志中点击 `https://xxxx.trycloudflare.com` 即可打开前端界面开始生成！


In [8]:
# @title 6. 启动 ComfyUI Web 服务（本地后台运行）
import subprocess
import time
import os
import socket

def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

os.chdir('/content/ComfyUI')
GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_H3_Outputs"

# 启动 ComfyUI 主程序
cmd_comfy = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram",
    "--output-directory", GDRIVE_OUTPUT_DIR
]

print("🚀 正在启动 ComfyUI 后台服务并加载模型配置...")
p_comfy = subprocess.Popen(cmd_comfy, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 轮询检测 ComfyUI 服务是否就绪
print("⏳ 正在等待 ComfyUI 本地端口 (8188) 响应...")
for i in range(45):
    if is_port_open(8188):
        print("🔥 ComfyUI 本地服务启动成功！已在后台安全运行。")
        print("👉 请直接运行下方单元格，在网页内嵌入加载 WebUI！")
        break
    time.sleep(1.0)
else:
    print("⚠️ 端口唤醒超时，请检查控制台或重新运行本单元格。")

🚀 正在启动 ComfyUI 后台服务并加载模型配置...
⏳ 正在等待 ComfyUI 本地端口 (8188) 响应...
🔥 ComfyUI 本地服务启动成功！已在后台安全运行。
👉 请直接运行下方单元格，在网页内嵌入加载 WebUI！


```markdown
### 🚀 极速通道：Colab 官方原生内嵌网页
运行下方单元格，无需额外安装或注册任何第三方隧道，直接在 Notebook 中打开交互式 ComfyUI Web 界面。
```

In [9]:
# @title 启动官方高速直连内嵌界面
from google.colab import output

try:
    print("\n" + "="*60)
    print("⚡ 正在嵌入 ComfyUI Web 交互界面...")
    print("💡 提示：如果下方加载出空白，请检查浏览器是否拦截了第三方 Cookie 或混合内容。")
    print("="*60 + "\n")

    # 在单元格内展示一个高度为 850px 的 ComfyUI 视窗，不依赖外部链接跳转，绝对防 404
    output.serve_kernel_port_as_iframe(8188, height='850')

except Exception as e:
    print(f"❌ 官方 Iframe 嵌入失败: {e}")


⚡ 正在嵌入 ComfyUI Web 交互界面...
💡 提示：如果下方加载出空白，请检查浏览器是否拦截了第三方 Cookie 或混合内容。



<IPython.core.display.Javascript object>

### 🌐 备用通道：Localtunnel 独立公网穿透
如果官方内嵌网页由于浏览器安全限制无法加载，可以使用 Localtunnel 方案获取一个可在浏览器标签页中直接打开的独立外链。

In [10]:
# @title 启动 Localtunnel 公网隧道
import urllib.request
import os

# 获取 Colab 实例的公网 IP，用于 Localtunnel 安全验证（Tunnel Password）
print("🔑 正在获取您的安全访问密码 (Tunnel Password)... ")
public_ip = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip()
print(f"📌 您的验证密码为: {public_ip}   <--- [请复制此 IP 填入打开的网页中]")

print("\n📦 正在全局安装 localtunnel 包...")
os.system("npm install -g localtunnel")

print("\n🚀 正在建立隧道并生成专属访问外链...")
# 启动 localtunnel 穿透 8188 端口
!lt --port 8188

🔑 正在获取您的安全访问密码 (Tunnel Password)... 
📌 您的验证密码为: 34.24.54.212   <--- [请复制此 IP 填入打开的网页中]

📦 正在全局安装 localtunnel 包...

🚀 正在建立隧道并生成专属访问外链...
your url is: https://honest-worlds-cheat.loca.lt
^C
